# Module 2: Deploy

> Part of the **Modular Workshops** series. Standalone, ~15 min.

Moving the agent to production requires **production-ready deployment infrastructure** — a managed runtime for the agent. **LangSmith Deployments** gives 30+ endpoints, persistence, HITL, and Studio out of the box.

This module ships the deep agent from Module 1 to LangSmith with the `langgraph` CLI.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

from utils.workshop import scoped, workshop_user

print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("TAVILY_API_KEY set:   ", bool(os.environ.get("TAVILY_API_KEY")))
print("Workshop user:        ", workshop_user())


---
# Part 1. Deploy — Ship the Agent

We deploy the agent in `agents/deep_agent/` to **LangSmith Deployments** with the `langgraph` CLI. Because `agents/deep_agent/agent.py` imports `model` from `utils.models`, whatever model configuration lives there ships with the image automatically — no extra flags, no separate config.


## 1.1 Project structure

A deployable LangGraph project is a directory with a `langgraph.json` config at the root that points at one or more graph objects. We already have one — `langgraph.json` at the workshop root registers the deep agent at `agents/deep_agent/agent.py`.

`dependencies: ["."]` tells the CLI to install this project — `pyproject.toml` at the config root — into the image.


In [ ]:
import os

agent_dir = str(project_root / "agents" / "deep_agent")
print("langgraph.json (workshop root)")
print("---")

for root, dirs, files in os.walk(agent_dir):
    # Skip __pycache__ for clarity
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


In [ ]:
# langgraph.json -- the deploy configuration
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

# AGENTS.md -- the agent's identity
agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


The agent itself lives in `agent.py`. The module-level `agent` variable is what gets deployed — `langgraph.json` references it as `"deep_agent": "./agents/deep_agent/agent.py:agent"`.


## 1.2 Local development

Three CLI commands you'll use (all run from the workshop root, where `langgraph.json` lives):

```bash
# Validate langgraph.json (imports each graph, checks deps)
langgraph validate

# Run locally for development (Studio UI + hot reload)
langgraph dev --port 2024

# Deploy to LangSmith (beta)
langgraph deploy
```

`langgraph dev` opens the LangGraph Studio UI in your browser — useful to step through tool calls and approve HITL interrupts visually. By default it connects to `https://smith.langchain.com` for the Studio frontend and talks to your local server.

**Docs:** [LangGraph CLI reference](https://docs.langchain.com/langsmith/langgraph-cli) · [Deploy on Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).


## 1.3 Validate the config

`langgraph validate` imports each graph in `langgraph.json` and checks the config — without building Docker or uploading anything. Use it to catch config and import errors before deploying.


In [ ]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
# !cd "{project_root}" && langgraph dev
print({project_root})

## 1.4 Deploy to LangSmith (optional)

Run the cell below to deploy to **LangSmith Deployments**. `langgraph deploy` builds a Docker image (locally if Docker is available, otherwise remotely on LangSmith's builder) and pushes it. Provisioning takes a few minutes.

> The image we're shipping picks up its model configuration from `utils/models.py` — no extra deploy flags needed.

> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. On Apple Silicon, local builds need Docker Buildx; without Docker, the CLI falls back to a remote build automatically.

> Everything in the project directory becomes part of the Docker build context. Add a `.dockerignore` to keep secrets out of the image — `langgraph deploy` reads `.env` from your local filesystem and uploads the values as deployment secrets, so the image itself never needs the file. Being in `.gitignore` does not exclude it.

Useful flags:
- `--name <name>` — deployment name (defaults to the project directory name)
- `--deployment-type dedicated` — always-on deployment (default is `serverless`; orgs on previous pricing use `dev`/`prod` until Oct 1, 2026)
- `--remote` — force remote build, skip local Docker
- `--no-wait` — return immediately rather than blocking on status

In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type prod` for production, or `--remote` to skip local Docker.
# Scoped per attendee so concurrent deploys don't target the same deployment.
import shutil, subprocess

workshop_user_name = workshop_user()
deployment_name = scoped(f"medical-order-processing-agent")
print("Deploying as:", deployment_name)

# subprocess with cwd= instead of a `cd && ...` shell magic: `&&` isn't valid
# in Windows PowerShell (pre-v7), and `!cd` runs in a throwaway subshell.
# Resolve the CLI's full path so it's found regardless of shell/PATHEXT.
LANGGRAPH = shutil.which("langgraph")
assert LANGGRAPH, "langgraph CLI not found on PATH — install with `uv pip install langgraph-cli`."
subprocess.run(
    [LANGGRAPH, "deploy", "--name", deployment_name, "--no-input"],
    cwd=str(project_root), check=True,
)


## 1.5 What you get with LangSmith Deployments

Once deployed, your agent is reachable through 30+ endpoints — you build it once, the platform exposes it everywhere:

| Capability | What you can do |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call your agent as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads (via the platform's Store) |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger your agent on a schedule |


---
# Part 2. A Lightweight React Frontend (runs from this notebook)

Part 1 shipped the agent to LangSmith. But you don't need a cloud deployment to
build and demo a UI — `langgraph dev` runs the *same graph* locally, exposing the
identical HTTP + streaming API at a fixed address (`http://127.0.0.1:2024`).

In this part the **notebook itself** scaffolds, wires, and launches a thin React
frontend for the medical device order desk — a chat panel where a coordinator asks
about payer policy, coding, and authorization for a device order and watches the
agent stream back. The cells below do the work; you just click the localhost link
at the end.

> **Local now, deployed later:** the exact same `App.tsx` works against your
> LangSmith deployment from Part 1 — swap one env var (`VITE_LANGGRAPH_API_URL`)
> from `http://127.0.0.1:2024` to your `https://<deployment>.us.langgraph.app` URL.

We use the official [`@langchain/langgraph-sdk`](https://docs.langchain.com/langgraph-platform/js-ts-sdk)
React bindings, so the browser talks to the graph server directly — no server we
have to write and maintain in the middle.

## 2.1 Prerequisites and constants

The frontend build needs Node + npm on your machine. This cell checks for them and
fixes the local addresses everything will use — no deployment URL to copy-paste,
because `langgraph dev` always serves on the same host and port.

In [ ]:
import shutil

# Fixed local addresses — no URL scraping needed.
API_URL = "http://127.0.0.1:2024"     # where `langgraph dev` serves the graph
FRONTEND_URL = "http://localhost:5173"  # Vite's default dev-server address
ASSISTANT_ID = "deep_agent"            # the graph id from langgraph.json
FRONTEND_DIR = project_root / "frontend"

node = shutil.which("node")
npm = shutil.which("npm")
print("node:", node or "NOT FOUND — install Node 18+ from https://nodejs.org")
print("npm: ", npm or "NOT FOUND")
print("API URL (langgraph dev):", API_URL)
print("Frontend URL (Vite):    ", FRONTEND_URL)
print("Frontend dir:           ", FRONTEND_DIR)

assert node and npm, "Node and npm are required for Part 2."

## 2.2 Allow the browser to call the local graph server

The React app runs on `localhost:5173` and calls the graph server on `127.0.0.1:2024`
— a different origin, so the browser enforces CORS. `langgraph dev` reads its CORS
policy from `langgraph.json` (`http.cors`). This cell adds `localhost:5173` to the
allowed origins **in place**, so the same config governs local dev and deployment.

In [ ]:
import json

langgraph_json_path = project_root / "langgraph.json"
config = json.loads(langgraph_json_path.read_text(encoding="utf-8"))

allowed = {FRONTEND_URL, "http://127.0.0.1:5173"}
http_cfg = config.setdefault("http", {})
cors = http_cfg.setdefault("cors", {})
origins = set(cors.get("allow_origins", []))
cors["allow_origins"] = sorted(origins | allowed)

# The SDK creates threads/runs with POST/PUT/PATCH/DELETE and sends a
# Content-Type header, so the preflight must allow them. The default is
# GET-only, which makes the browser reject the preflight (400 "Disallowed
# CORS method") before any real request goes out.
cors["allow_methods"] = ["GET", "POST", "PUT", "PATCH", "DELETE", "OPTIONS"]
cors["allow_headers"] = ["*"]

langgraph_json_path.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
print("Updated langgraph.json http.cors:")
print(json.dumps(config["http"], indent=2))
print()
print("NOTE: restart `langgraph dev` (re-run cell 2.3 after stopping it) so the")
print("new CORS policy takes effect — the server reads this config at startup.")

## 2.3 Start the local graph server (`langgraph dev`)

This launches `langgraph dev` as a **background process** so the notebook keeps
running. It serves the deep agent — same tools, subagent, memory, and HITL — at
`http://127.0.0.1:2024`. We poll `/ok` until it's healthy.

> Re-running this cell won't start a second server: it detects the healthy one and
> reuses it. To stop it later, run the teardown cell at the end of Part 2.

In [ ]:
import shutil, subprocess, time, urllib.request, urllib.error

# Resolve the CLI's full path so it's found regardless of shell/PATHEXT (Windows).
LANGGRAPH = shutil.which("langgraph")
assert LANGGRAPH, "langgraph CLI not found on PATH."

def _server_healthy(url):
    try:
        with urllib.request.urlopen(f"{url}/ok", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if _server_healthy(API_URL):
    print(f"langgraph dev already running at {API_URL}")
    dev_server = globals().get("dev_server")
else:
    # --no-browser: don't pop Studio; we just want the API.
    # --no-reload: the dev server writes checkpoint state under .langgraph_api/,
    #   which its own file watcher would then detect — a self-triggering reload
    #   loop that drops in-flight requests and makes /threads appear to hang.
    #   Disabling the watcher is the right call for a demo server anyway.
    # Log to a file, NOT an unread PIPE: a full pipe buffer would block the
    #   server process mid-write and wedge it.
    dev_log = open(project_root / "langgraph_dev.log", "w", encoding="utf-8")
    dev_server = subprocess.Popen(
        [LANGGRAPH, "dev", "--no-browser", "--no-reload", "--port", "2024"],
        cwd=str(project_root),
        stdout=dev_log, stderr=subprocess.STDOUT, text=True,
    )
    print("Starting langgraph dev … (logs: langgraph_dev.log)")
    for _ in range(60):
        if _server_healthy(API_URL):
            break
        if dev_server.poll() is not None:
            raise RuntimeError(
                "langgraph dev exited early — see langgraph_dev.log:\n"
                + (project_root / "langgraph_dev.log").read_text(encoding="utf-8")
            )
        time.sleep(1)
    assert _server_healthy(API_URL), "langgraph dev did not become healthy in time."
    print(f"langgraph dev is up at {API_URL}")

## 2.4 Scaffold the React app

This cell creates a minimal Vite + React-TypeScript app in `frontend/` and installs
the one runtime dependency we need — the LangGraph SDK. It's idempotent: if
`frontend/` already exists it skips the scaffold and just ensures dependencies.

The `@langchain/langgraph-sdk/react` entrypoint ships a `useStream` hook that handles
the run lifecycle — creating a thread, streaming tokens, surfacing tool calls, and
resuming HITL interrupts — so the component stays tiny.

In [ ]:
import shutil, subprocess

# On Windows, `npm` is `npm.cmd` (a batch shim), not a real .exe. subprocess
# with a list and no shell=True calls CreateProcess directly, which does NOT
# resolve PATHEXT — so bare ["npm", ...] raises WinError 2. Resolving the full
# path with shutil.which() (which honors PATHEXT) works on every OS.
NPM = shutil.which("npm")
assert NPM, "npm not found on PATH — install Node 18+ from https://nodejs.org"

def run(cmd, cwd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=str(cwd), check=True)

if not FRONTEND_DIR.exists():
    # `npm create vite` scaffolds into ./frontend from the project root.
    run([NPM, "create", "vite@latest", "frontend", "--",
         "--template", "react-ts"], project_root)
else:
    print(f"{FRONTEND_DIR} already exists — skipping scaffold.")

run([NPM, "install"], FRONTEND_DIR)
run([NPM, "install", "@langchain/langgraph-sdk"], FRONTEND_DIR)
print("Frontend dependencies installed.")

## 2.5 Wire the app to the local server and write the chat component

Now the notebook writes the frontend files into `frontend/`:

- **`.env.local`** — points Vite at the local graph server (`API_URL`) and the graph
  id. Swap `VITE_LANGGRAPH_API_URL` for your LangSmith deployment URL to run against
  the cloud instead — nothing else changes.
- **`src/App.tsx`** — a styled chat panel for the device order desk. The `useStream`
  hook handles threads, streaming, and HITL; the component normalizes the agent's
  rich messages into **chat bubbles** (user vs. assistant) and renders tool calls as
  subtle chips, instead of dumping the raw streamed content blocks.
- **`src/App.css`** styling: message bubbles, a typing indicator, tool-call chips,
  an empty-state with suggested prompts, and light/dark support.

In [ ]:
# .env.local — Vite reads VITE_-prefixed vars at build time.
env_local = FRONTEND_DIR / ".env.local"
env_local.write_text(
    f'VITE_LANGGRAPH_API_URL="{API_URL}"\n'
    f'VITE_LANGGRAPH_ASSISTANT_ID="{ASSISTANT_ID}"\n',
    encoding="utf-8",
)
print("Wrote", env_local)

# The chat UI: App.tsx renders human/assistant bubbles and shows tool calls as
# chips, instead of dumping raw streamed content. App.css styles it.
app_tsx = r"""
import { useStream } from "@langchain/langgraph-sdk/react";
import { useEffect, useMemo, useRef, useState } from "react";
import "./App.css";

// The agent returns rich messages: `ai` content is an array of blocks
// (text / reasoning / function_call), `tool` messages carry a tool result,
// and `human` content is a plain string. We normalize each message into a
// simple shape the chat UI can render.
type ContentBlock =
  | { type: "text"; text: string }
  | { type: "reasoning"; [k: string]: unknown }
  | { type: "function_call"; name?: string; [k: string]: unknown }
  | { type: string; [k: string]: unknown };

type RawMessage = {
  id?: string;
  type: "human" | "ai" | "tool" | "system";
  content: string | ContentBlock[];
  name?: string;
  tool_calls?: { name?: string }[];
};

type ChatItem = {
  key: string;
  role: "user" | "assistant";
  text: string;
  toolCalls: string[];
};

const SUGGESTIONS = [
  "What auth is needed for an insulin pump order under Medicare?",
  "Which CPT/HCPCS codes apply to a continuous glucose monitor?",
  "Draft an exception note for a denied nebulizer order.",
];

function textFromContent(content: string | ContentBlock[]): string {
  if (typeof content === "string") return content;
  return content
    .filter((b) => b.type === "text" && typeof (b as { text?: unknown }).text === "string")
    .map((b) => (b as { text: string }).text)
    .join("\n")
    .trim();
}

function toolCallsFromMessage(m: RawMessage): string[] {
  const fromField = (m.tool_calls ?? []).map((t) => t.name).filter(Boolean) as string[];
  const fromBlocks = Array.isArray(m.content)
    ? m.content
        .filter((b) => b.type === "function_call")
        .map((b) => (b as { name?: string }).name)
        .filter(Boolean)
    : [];
  return [...fromField, ...(fromBlocks as string[])];
}

// Collapse the raw message list into displayable chat turns:
// user bubbles, assistant text bubbles, and "used tool X" chips.
// Tool result messages and pure reasoning are intentionally hidden.
function toChatItems(messages: RawMessage[]): ChatItem[] {
  const items: ChatItem[] = [];
  messages.forEach((m, i) => {
    if (m.type === "human") {
      const text = textFromContent(m.content);
      if (text) items.push({ key: m.id ?? `u-${i}`, role: "user", text, toolCalls: [] });
    } else if (m.type === "ai") {
      const text = textFromContent(m.content);
      const toolCalls = toolCallsFromMessage(m);
      if (text || toolCalls.length) {
        items.push({ key: m.id ?? `a-${i}`, role: "assistant", text, toolCalls });
      }
    }
    // `tool` and `system` messages are omitted from the chat transcript.
  });
  return items;
}

function prettyToolName(name: string): string {
  const map: Record<string, string> = {
    task: "delegating to research agent",
    tavily_search: "searching the web",
    easy_search: "searching the web",
    write_file: "drafting a note",
    edit_file: "editing a note",
    read_file: "reading a file",
    ls: "listing files",
  };
  return map[name] ?? name.replace(/_/g, " ");
}

export default function App() {
  const [input, setInput] = useState("");
  const scrollRef = useRef<HTMLDivElement>(null);

  const thread = useStream<{ messages: RawMessage[] }>({
    apiUrl: import.meta.env.VITE_LANGGRAPH_API_URL,
    assistantId: import.meta.env.VITE_LANGGRAPH_ASSISTANT_ID,
    messagesKey: "messages",
  });

  const items = useMemo(
    () => toChatItems((thread.messages as RawMessage[]) ?? []),
    [thread.messages],
  );

  // Auto-scroll to the newest message.
  useEffect(() => {
    scrollRef.current?.scrollTo({ top: scrollRef.current.scrollHeight, behavior: "smooth" });
  }, [items, thread.isLoading]);

  const submit = (value: string) => {
    const text = value.trim();
    if (!text || thread.isLoading) return;
    thread.submit({ messages: [{ type: "human", content: text }] });
    setInput("");
  };

  const onSubmit = (e: React.FormEvent) => {
    e.preventDefault();
    submit(input);
  };

  const empty = items.length === 0;

  return (
    <div className="chat">
      <header className="chat-header">
        <div className="brand">
          <span className="brand-dot" />
          <div>
            <h1>Device Order Desk</h1>
            <p className="subtitle">
              Payer policy, coding &amp; authorization for medical device orders
            </p>
          </div>
        </div>
      </header>

      <div className="messages" ref={scrollRef}>
        {empty && (
          <div className="empty">
            <h2>How can I help with this order?</h2>
            <p className="empty-sub">
              Ask about coverage, prior authorization, or coding requirements.
            </p>
            <div className="suggestions">
              {SUGGESTIONS.map((s) => (
                <button key={s} className="suggestion" onClick={() => submit(s)}>
                  {s}
                </button>
              ))}
            </div>
          </div>
        )}

        {items.map((item) => (
          <div key={item.key} className={`row ${item.role}`}>
            {item.role === "assistant" && <div className="avatar">AI</div>}
            <div className="bubble-wrap">
              {item.toolCalls.length > 0 && (
                <div className="tools">
                  {item.toolCalls.map((name, i) => (
                    <span key={`${name}-${i}`} className="tool-chip">
                      <span className="spinner-dot" />
                      {prettyToolName(name)}
                    </span>
                  ))}
                </div>
              )}
              {item.text && (
                <div className={`bubble ${item.role}`}>
                  {item.text.split("\n").map((line, i) => (
                    <p key={i}>{line || "\u00a0"}</p>
                  ))}
                </div>
              )}
            </div>
          </div>
        ))}

        {thread.isLoading && (
          <div className="row assistant">
            <div className="avatar">AI</div>
            <div className="bubble assistant typing">
              <span className="dot" />
              <span className="dot" />
              <span className="dot" />
            </div>
          </div>
        )}
      </div>

      <form className="composer" onSubmit={onSubmit}>
        <input
          value={input}
          onChange={(e) => setInput(e.target.value)}
          placeholder="Ask about a device order…"
          aria-label="Message"
        />
        {thread.isLoading ? (
          <button type="button" className="stop" onClick={() => thread.stop()}>
            Stop
          </button>
        ) : (
          <button type="submit" className="send" disabled={!input.trim()}>
            Send
          </button>
        )}
      </form>
    </div>
  );
}
"""

app_css = r"""
/* Override index.css #root defaults for a full-height chat layout. */
#root {
  width: 100%;
  max-width: 820px;
  text-align: left;
  border-inline: 1px solid var(--border);
}

.chat {
  display: flex;
  flex-direction: column;
  height: 100svh;
  min-height: 0;
}

/* Header */
.chat-header {
  padding: 16px 24px;
  border-bottom: 1px solid var(--border);
  background: var(--bg);
}
.brand {
  display: flex;
  align-items: center;
  gap: 12px;
}
.brand-dot {
  width: 12px;
  height: 12px;
  border-radius: 50%;
  background: var(--accent);
  box-shadow: 0 0 0 4px var(--accent-bg);
  flex: none;
}
.chat-header h1 {
  font-size: 20px;
  letter-spacing: -0.2px;
  margin: 0;
}
.subtitle {
  font-size: 13px;
  color: var(--text);
  margin: 2px 0 0;
}

/* Message list */
.messages {
  flex: 1;
  min-height: 0;
  overflow-y: auto;
  padding: 24px;
  display: flex;
  flex-direction: column;
  gap: 18px;
  scroll-behavior: smooth;
}

/* Empty state */
.empty {
  margin: auto;
  text-align: center;
  max-width: 520px;
}
.empty h2 {
  font-size: 26px;
  color: var(--text-h);
  margin-bottom: 6px;
}
.empty-sub {
  color: var(--text);
  margin-bottom: 24px;
}
.suggestions {
  display: flex;
  flex-direction: column;
  gap: 10px;
}
.suggestion {
  border: 1px solid var(--border);
  background: var(--bg);
  color: var(--text-h);
  border-radius: 12px;
  padding: 12px 16px;
  font-size: 15px;
  text-align: left;
  cursor: pointer;
  transition: border-color 0.15s, background 0.15s, transform 0.05s;
}
.suggestion:hover {
  border-color: var(--accent-border);
  background: var(--accent-bg);
}
.suggestion:active {
  transform: translateY(1px);
}

/* Rows + avatar */
.row {
  display: flex;
  gap: 12px;
  max-width: 100%;
}
.row.user {
  justify-content: flex-end;
}
.row.assistant {
  justify-content: flex-start;
}
.avatar {
  width: 32px;
  height: 32px;
  border-radius: 50%;
  flex: none;
  display: grid;
  place-items: center;
  font-size: 12px;
  font-weight: 600;
  color: #fff;
  background: var(--accent);
  margin-top: 2px;
}

.bubble-wrap {
  display: flex;
  flex-direction: column;
  gap: 6px;
  max-width: 78%;
}

/* Bubbles */
.bubble {
  padding: 12px 16px;
  border-radius: 16px;
  font-size: 15.5px;
  line-height: 1.5;
  word-wrap: break-word;
  overflow-wrap: anywhere;
}
.bubble p {
  margin: 0;
}
.bubble p + p {
  margin-top: 8px;
}
.bubble.user {
  background: var(--accent);
  color: #fff;
  border-bottom-right-radius: 4px;
}
.bubble.assistant {
  background: var(--code-bg);
  color: var(--text-h);
  border: 1px solid var(--border);
  border-bottom-left-radius: 4px;
}

/* Tool-call chips */
.tools {
  display: flex;
  flex-wrap: wrap;
  gap: 6px;
}
.tool-chip {
  display: inline-flex;
  align-items: center;
  gap: 7px;
  font-size: 12.5px;
  color: var(--accent);
  background: var(--accent-bg);
  border: 1px solid var(--accent-border);
  border-radius: 999px;
  padding: 4px 11px;
  text-transform: lowercase;
}
.spinner-dot {
  width: 6px;
  height: 6px;
  border-radius: 50%;
  background: var(--accent);
}

/* Typing indicator */
.bubble.typing {
  display: inline-flex;
  gap: 5px;
  align-items: center;
  padding: 14px 16px;
}
.dot {
  width: 7px;
  height: 7px;
  border-radius: 50%;
  background: var(--text);
  opacity: 0.5;
  animation: blink 1.3s infinite ease-in-out both;
}
.dot:nth-child(2) {
  animation-delay: 0.2s;
}
.dot:nth-child(3) {
  animation-delay: 0.4s;
}
@keyframes blink {
  0%,
  80%,
  100% {
    opacity: 0.25;
    transform: translateY(0);
  }
  40% {
    opacity: 1;
    transform: translateY(-2px);
  }
}

/* Composer */
.composer {
  display: flex;
  gap: 10px;
  padding: 16px 24px 22px;
  border-top: 1px solid var(--border);
  background: var(--bg);
}
.composer input {
  flex: 1;
  padding: 13px 16px;
  font-size: 15.5px;
  font-family: var(--sans);
  color: var(--text-h);
  background: var(--bg);
  border: 1px solid var(--border);
  border-radius: 14px;
  outline: none;
  transition: border-color 0.15s, box-shadow 0.15s;
}
.composer input:focus {
  border-color: var(--accent-border);
  box-shadow: 0 0 0 3px var(--accent-bg);
}
.composer button {
  border: none;
  border-radius: 14px;
  padding: 0 22px;
  font-size: 15px;
  font-weight: 600;
  cursor: pointer;
  transition: opacity 0.15s, transform 0.05s;
}
.composer button:active {
  transform: translateY(1px);
}
.send {
  background: var(--accent);
  color: #fff;
}
.send:disabled {
  opacity: 0.4;
  cursor: not-allowed;
}
.stop {
  background: var(--code-bg);
  color: var(--text-h);
  border: 1px solid var(--border) !important;
}
"""

app_path = FRONTEND_DIR / "src" / "App.tsx"
app_path.write_text(app_tsx.lstrip(), encoding="utf-8")
print("Wrote", app_path)

css_path = FRONTEND_DIR / "src" / "App.css"
css_path.write_text(app_css.lstrip(), encoding="utf-8")
print("Wrote", css_path)

## 2.6 Launch the frontend

This starts the Vite dev server as a background process and prints the link. Open it
and start chatting — each message opens (or reuses) a thread on the local graph
server and streams the agent's response token-by-token.

Because the agent ships with HITL on `write_file`/`edit_file`, `useStream` also
exposes `thread.interrupt` and `thread.submit({}, { command: { resume: ... } })` when
you want the coordinator to approve a drafted exception note before it's written.

In [ ]:
import shutil, subprocess, time, urllib.request

# Resolve npm's full path so this works on Windows (npm.cmd) too, even if
# cell 2.4 wasn't run in this session.
NPM = globals().get("NPM") or shutil.which("npm")

def _url_up(url):
    try:
        with urllib.request.urlopen(url, timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if _url_up(FRONTEND_URL):
    print(f"Frontend already running at {FRONTEND_URL}")
    frontend_server = globals().get("frontend_server")
else:
    # Log to a file, not an unread PIPE (a full pipe buffer would wedge Vite).
    vite_log = open(FRONTEND_DIR / "vite_dev.log", "w", encoding="utf-8")
    frontend_server = subprocess.Popen(
        [NPM, "run", "dev", "--", "--port", "5173", "--strictPort"],
        cwd=str(FRONTEND_DIR),
        stdout=vite_log, stderr=subprocess.STDOUT, text=True,
    )
    print("Starting Vite dev server … (logs: frontend/vite_dev.log)")
    for _ in range(60):
        if _url_up(FRONTEND_URL):
            break
        if frontend_server.poll() is not None:
            raise RuntimeError(
                "Vite exited early — see frontend/vite_dev.log:\n"
                + (FRONTEND_DIR / "vite_dev.log").read_text(encoding="utf-8")
            )
        time.sleep(1)

print()
print("=" * 50)
print(f"  Open the Device Order Desk:  {FRONTEND_URL}")
print("=" * 50)

## 2.7 Best practice: how the frontend should connect

Whether local (`langgraph dev`) or deployed (LangSmith), the graph server **is** your
backend — REST, streaming, persistence, HITL. Don't rebuild that layer. Keep the
connection as direct as your requirements allow.

**Best practice**
- **Call the deployment directly from the frontend for chat/streaming.** The LangGraph
  SDK's `useStream` talks to the graph server's REST + streaming API from the browser —
  no server code in between to write or operate.
- **Add a thin backend/proxy only if you need auth/session handling, CORS control, or
  to keep secrets off the client.** It stays a pass-through: mint or validate a session,
  attach the API key server-side, forward the request. It is not a place for business
  logic.
- **Use agent middleware for agent *behavior* (logging, validation, retries, tool
  policy) — not as a required UI layer.** Behavior belongs in the graph so every caller
  (this UI, an MCP client, a cron run) gets it. The UI shouldn't depend on middleware to
  function.

In short: the browser → graph-server path is the default. Reach for a proxy only for
auth, CORS, or secret-hiding — and put agent behavior in middleware on the graph, where
it applies to every client, not in the frontend.

## 2.8 Teardown (optional)

Run this to stop the background servers when you're done with the demo.

In [ ]:
for name in ("frontend_server", "dev_server"):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        print(f"Stopped {name}")
    else:
        print(f"{name}: not running")

## Recap

| What | How |
|---|---|
| Deployable graph config | `langgraph.json` at repo root |
| Agent identity + skills | `AGENTS.md`, `skills/` |
| Validate before shipping | `langgraph validate` |
| Ship it | `langgraph deploy --name <your-deployment-name>` |
| Lightweight UI | React + `@langchain/langgraph-sdk` `useStream`; the notebook scaffolds, wires, and launches it against `langgraph dev` |
| Local vs. deployed | Same `App.tsx`; swap `VITE_LANGGRAPH_API_URL` from `127.0.0.1:2024` to your deployment URL |
| Connection best practice | Browser → graph server by default; thin proxy only for auth/CORS/secrets; middleware for behavior |

**The production pattern:** one config file + one deploy command. The agent now lives behind a managed server, with 30+ endpoints available to call it — including from a thin React frontend that streams straight from the browser.

**Next:** Module 3 — LangSmith (prompt engineering, tracing, querying traces, offline + online evals, annotation queues).
